# i2b2 ETL API Client
Connect to the Flask app running on port 5000.

**Auth format:** `Basic base64(project\\username:session_key)`  
The app expects the username as `Project\\Username` and the password as the i2b2 session key.

In [1]:
import requests
import json
import base64

# --- Config ---
BASE_URL = "http://localhost:5000"
PROJECT  = "Demo"        # i2b2 project name
USERNAME = "demo"        # i2b2 username
PASSWORD = "demouser"    # i2b2 password (will be exchanged for a session key)

# i2b2 PM service (used to get a session key)
PM_SERVICE_URL = "http://localhost:8080/i2b2/services/PMService/getServices"

ModuleNotFoundError: No module named 'requests'

## 1. Get Session Key from i2b2 PM Service

In [ ]:
def get_session_key(pm_url, username, password):
    """Exchange username+password for an i2b2 session key."""
    xml = f"""<?xml version=\"1.0\" encoding=\"UTF-8\"?>
<i2b2:request xmlns:i2b2=\"http://www.i2b2.org/xsd/hive/msg/1.1/\">
  <message_header>
    <security>
      <domain>i2b2demo</domain>
      <username>{username}</username>
      <password>{password}</password>
    </security>
  </message_header>
  <message_body/>
</i2b2:request>"""

    headers = {
        "Content-Type": "application/xml",
        "Accept": "*/*"
    }
    response = requests.post(pm_url, data=xml, headers=headers).text
    for part in response.split("</password>"):
        if "SessionKey:" in part:
            return part[part.find("SessionKey:") + len("SessionKey:"):].strip()
    raise ValueError("Session key not found in PM response. Check credentials.")


session_key = get_session_key(PM_SERVICE_URL, USERNAME, PASSWORD)
print(f"Session key obtained: {session_key[:20]}...")

## 2. Build Auth Headers

The Flask app expects Basic auth where:
- **username** = `Project\Username`  (one backslash)
- **password** = the session key

In [ ]:
def make_headers(project, username, session_key):
    """Build HTTP Basic Auth + project headers for the i2b2 ETL API."""
    credentials = f"{project}\\{username}:{session_key}"
    encoded = base64.b64encode(credentials.encode("ascii")).decode("ascii")
    return {
        "Authorization": f"Basic {encoded}",
        "X-Project-Name": project,
        "Content-Type": "application/json",
    }

HEADERS = make_headers(PROJECT, USERNAME, session_key)
print("Headers ready.")

## 3. Helper — Pretty Print Responses

In [ ]:
def show(response):
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)

---
## 4. Concepts API

In [ ]:
# GET all concepts
r = requests.get(f"{BASE_URL}/etl/concepts", headers=HEADERS)
show(r)

In [ ]:
# GET concept by human path
r = requests.get(f"{BASE_URL}/etl/concepts", headers=HEADERS, params={"hpath": "/ML/Diagnosis/DM2"})
show(r)

In [ ]:
# POST — create a concept
concept = {
    "code": "Diabetes_mellitus_type2_code",
    "path": "/ML/Diagnosis/DM2",
    "type": "assertion"
}
r = requests.post(f"{BASE_URL}/etl/concepts", headers=HEADERS, json=concept)
show(r)

---
## 5. ML — Build Model

In [ ]:
# POST /etl/ml_build_model — create ML concept with training config
ml_concept = {
    "code": "Diabetes_mellitus_type2_code",
    "description": "Model trained on Diabetes patients",
    "path": "/ML/Diagnosis/DM2",
    "blob": {
        "positive_patient_set": ["positive_dm2"],
        "negative_patient_set": ["negative_dm2"],
        "data_paths": ["diabetes/diabetes_ehr1", "diabetes/diabetes_ehr2"],
        "label_paths": ["diabetes/label"],
        "data_period_start": "1992-02-02",
        "data_period_end": "2024-08-20"
    }
}
r = requests.post(f"{BASE_URL}/etl/ml_build_model", headers=HEADERS, json=ml_concept)
show(r)

---
## 6. Jobs API

In [ ]:
# POST — submit a build-model job
job = {
    "jobType": "ml-build",
    "input": {"path": "/ML/Diagnosis/DM2"}
}
r = requests.post(f"{BASE_URL}/etl/job", headers=HEADERS, json=job)
show(r)

In [ ]:
# GET — check job status
JOB_ID = 1  # replace with the job ID returned above
r = requests.get(f"{BASE_URL}/etl/job", headers=HEADERS, params={"jobId": JOB_ID})
show(r)

In [ ]:
# POST — update job status (e.g. move from PRE to PENDING)
r = requests.post(
    f"{BASE_URL}/etl/update_job",
    headers=HEADERS,
    json={"jobId": JOB_ID, "pre": "COMPLETED", "post": "PENDING"}
)
show(r)

---
## 7. ML — Apply Model

In [ ]:
# POST /etl/ml_apply_model
apply_payload = {
    "path": "/ML/Diagnosis/DM2",
    "target_path": "/ML/Diagnosis/DM2/target"
}
r = requests.post(f"{BASE_URL}/etl/ml_apply_model", headers=HEADERS, json=apply_payload)
show(r)

---
## 8. Facts API

In [ ]:
# GET facts by concept path
r = requests.get(f"{BASE_URL}/etl/facts", headers=HEADERS, params={"hpath": "/ML/Diagnosis/DM2"})
show(r)

In [ ]:
# DELETE facts by concept path
r = requests.delete(f"{BASE_URL}/etl/facts", headers=HEADERS, params={"hpath": "/ML/Diagnosis/DM2"})
show(r)

---
## 9. ETL Data — Load / Delete

In [ ]:
import os

# POST — upload CSV files to load data
csv_path = "../sample_files/concepts.csv"   # change to your file path

upload_headers = {k: v for k, v in HEADERS.items() if k != "Content-Type"}  # let requests set multipart

with open(csv_path, "rb") as f:
    r = requests.post(
        f"{BASE_URL}/etl/data",
        headers=upload_headers,
        files={"files": (os.path.basename(csv_path), f, "text/csv")},
        params={"loginProject": PROJECT}
    )
show(r)

In [ ]:
# GET — fetch ETL log after a load
r = requests.get(f"{BASE_URL}/etl/logs", headers=HEADERS, params={"loginProject": PROJECT})
show(r)

---
## 10. Misc

In [ ]:
# Config / institution info (no auth required)
r = requests.get(f"{BASE_URL}/etl/configData")
show(r)

In [ ]:
# Check DB connectivity
r = requests.get(f"{BASE_URL}/etl/checkUserDatabase", headers=HEADERS, params={"loginProject": PROJECT})
show(r)